In [54]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import FunctionTransformer
import joblib

# **Feature Engineering**

In [55]:
df = pd.read_csv('../data/train.csv')
df.drop(columns=['Unnamed: 0', 'id'], inplace=True, errors='ignore')

In [56]:
def combine_delays(X):
    X = X.copy()
    X['Arrival Delay in Minutes'] = X['Arrival Delay in Minutes'].fillna(0)
    X['Total_Delay'] = X['Departure Delay in Minutes'] + X['Arrival Delay in Minutes']
    return X.drop(['Departure Delay in Minutes', 'Arrival Delay in Minutes'], axis=1)

In [57]:
delay_transformer = FunctionTransformer(combine_delays)

I fill NaN with 0 and combined both features related to delays because of strong correlation between them.

## Creating piplines

In [58]:
cat_features = ['Gender', 'Customer Type', 'Type of Travel', 'Class']
cat_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

In [59]:
num_features = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
columns_to_remove = ['Departure Delay in Minutes', 'Arrival Delay in Minutes']
num_features = [col for col in num_features if col not in columns_to_remove]
num_features.append('Total_Delay')
num_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

In [60]:
preprocessor_core = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_features),
        ('cat', cat_transformer, cat_features)
    ])

In [61]:
preprocessor = Pipeline(steps=[
    ('delays', delay_transformer),
    ('core', preprocessor_core)
])

### Split and preprocessing on dataset

In [62]:
X = df.drop('satisfaction', axis=1)
y = df['satisfaction'].map({'neutral or dissatisfied': 0, 'satisfied': 1})
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [63]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

### Safe dataset after Feature Engineering

In [64]:
joblib.dump((X_train, X_test, y_train, y_test), '../data/processed_data.pkl')
joblib.dump(preprocessor, '../models/preprocessor.pkl')

['../models/preprocessor.pkl']